# CHARTA Layer 4 — GNN Training for Readmission Risk Prediction

**Colab notebook (Drive-only, no GitHub)** — trains [`ClinicalGraphSAGE`](src/layer4/graph_model.py:21) + [`ReadmissionHead`](src/layer4/readmission_head.py:17) on T4 GPU.

## Architecture
```
Patient HeteroData Graph
        │
        ▼
ClinicalGraphSAGE (2-layer GraphSAGE, 768→256→256)
        │
        ▼
global_mean_pool → patient embedding [256]
        │
        ▼
ReadmissionHead (256→128→1 logit)
        │
        ▼
sigmoid → risk probability [0,1]
```

## Google Drive Folder Structure Required
```
MyDrive/
  CHARTA/
    src/
      __init__.py              # (empty)
      shared/
        __init__.py            # (empty)
        utils.py               # save_json / load_json helpers
      layer4/
        __init__.py            # (empty)
        config.py              # hyperparameters
        clinical_dataset.py    # ClinicalGraphDataset
        graph_model.py         # ClinicalGraphSAGE encoder
        readmission_head.py    # ReadmissionHead + ReadmissionRiskModel
        trainer.py             # train_epoch / evaluate / train()
        pipeline.py            # batch inference
    data/
      corpus_labels.csv        # patient_id,readmission,deterioration,medication
      mtsamples_graphs/        # *_graph.pt + *_graph_meta.json
      openI_graphs/            # *_graph.pt files
```

## Output
- Checkpoint: `models/best_readmission_model.pt` (saved to Drive)
- Predictions: `data/predictions/*_predictions.json`

## 1. Mount Google Drive & Define Paths

In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# ── ALL paths defined HERE — the single place to adjust ─────────────
DRIVE_BASE = "/content/drive/MyDrive/CHARTA"

# Source code (copied to Colab local runtime)
DRIVE_SRC    = f"{DRIVE_BASE}/src"
LOCAL_SRC    = "/content/CHARTA/src"

# Data (read directly from Drive — no copying needed)
GRAPHS_FOLDER = [
    f"{DRIVE_BASE}/data/mtsamples_graphs",
    f"{DRIVE_BASE}/data/openI_graphs",
]
LABELS_CSV = f"{DRIVE_BASE}/data/corpus_labels.csv"

# Outputs (write back to Drive so checkpoints persist)
CHECKPOINT_DIR = f"{DRIVE_BASE}/models"
PREDICTIONS_DIR = f"{DRIVE_BASE}/data/predictions"

import os
from pathlib import Path

print(f"Drive base:     {DRIVE_BASE}")
print(f"Graph folders:  {GRAPHS_FOLDER}")
print(f"Labels CSV:     {LABELS_CSV}")
print(f"Checkpoints →:  {CHECKPOINT_DIR}")
print(f"Predictions →:  {PREDICTIONS_DIR}")

# ── Validate that the Drive folder exists ───────────────────────────
if not Path(DRIVE_BASE).exists():
    print(f"\n*** ERROR: {DRIVE_BASE} not found! ***")
    print("Ensure your Google Drive has a CHARTA/ folder at the root.")
else:
    print(f"\n✓ Drive CHARTA/ folder found.")

## 2. Copy Source Files from Drive to Colab Runtime

Colab can't import Python modules directly from Drive (the path is too slow & unreliable). We copy the `src/` tree to the local Colab runtime.

In [ ]:
# @title Copy src/ from Drive to /content/CHARTA/src/
import shutil

if Path(DRIVE_SRC).exists():
    if Path(LOCAL_SRC).exists():
        shutil.rmtree(LOCAL_SRC)
    shutil.copytree(DRIVE_SRC, LOCAL_SRC)
    print(f"✓ Copied {DRIVE_SRC} → {LOCAL_SRC}")

    # Verify key files exist
    required = [
        "layer4/config.py",
        "layer4/clinical_dataset.py",
        "layer4/graph_model.py",
        "layer4/readmission_head.py",
        "layer4/trainer.py",
        "layer4/pipeline.py",
        "shared/utils.py",
    ]
    missing = [f for f in required if not (Path(LOCAL_SRC) / f).exists()]
    if missing:
        print(f"\n*** WARNING: Missing source files: {missing} ***")
        print("Upload them to your Drive CHARTA/src/ folder.")
    else:
        print("✓ All source files present.")
else:
    print(f"*** ERROR: {DRIVE_SRC} not found on Drive! ***")
    print("Upload the CHARTA/src/ folder to Google Drive before running.")

In [ ]:
# @title Add /content/CHARTA/src to sys.path
import sys
sys.path.insert(0, LOCAL_SRC)
print(f"sys.path includes: {LOCAL_SRC}")

# Quick import test (before heavy deps install)
try:
    from shared.utils import save_json, load_json
    print("✓ shared.utils imported OK")
except Exception as e:
    print(f"✗ shared.utils import failed: {e}")

## 3. Install Dependencies

In [ ]:
# @title Install PyTorch Geometric + other dependencies
# PyG must match the installed PyTorch version. This cell detects the
# CUDA/PyTorch version and installs the correct PyG wheel.

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")
    !nvidia-smi --query-gpu=name --format=csv,noheader

# Install PyG (matches torch 2.x + CUDA 12.x on T4)
# Pre-extract torch version for reliable IPython shell interpolation
tv = torch.__version__
!pip install -q torch-scatter torch-sparse torch-cluster \
    -f https://data.pyg.org/whl/torch-{tv}.html
!pip install -q torch-geometric

# Other deps
!pip install -q scikit-learn pandas matplotlib seaborn tqdm

# Verify PyG installed correctly before proceeding
import torch_geometric
print(f"PyTorch Geometric version: {torch_geometric.__version__}")
print("\n✓ All dependencies installed.")

## 4. Imports & Config

In [ ]:
# @title Import all Layer 4 modules
from __future__ import annotations

import logging
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import seaborn as sns

# Layer 4 modules (from Drive-copied src/)
from layer4.config import (
    GRAPHSAGE_IN_DIM,
    GRAPHSAGE_HIDDEN_DIM,
    GRAPHSAGE_OUT_DIM,
    GRAPHSAGE_NUM_LAYERS,
    GRAPHSAGE_DROPOUT,
    LEARNING_RATE,
    WEIGHT_DECAY,
    NUM_EPOCHS,
    BATCH_SIZE,
    POSITIVE_CLASS_WEIGHT,
    RANDOM_SEED,
    GRAD_CLIP_MAX_NORM,
    RISK_THRESHOLD,
)
from layer4.clinical_dataset import ClinicalGraphDataset, collate_fn
from layer4.graph_model import ClinicalGraphSAGE
from layer4.readmission_head import ReadmissionHead, ReadmissionRiskModel
from layer4.trainer import train_epoch, evaluate, _split_dataset, _set_seed

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("layer4_notebook")

# Pretty printing
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✓ All imports OK")
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")

In [ ]:
# @title Print hyperparameter summary (from layer4/config.py on Drive)
print("=" * 50)
print("LAYER 4 — HYPERPARAMETER SUMMARY")
print("=" * 50)
print(f"  GraphSAGE input dim:      {GRAPHSAGE_IN_DIM}")
print(f"  GraphSAGE hidden dim:     {GRAPHSAGE_HIDDEN_DIM}")
print(f"  GraphSAGE output dim:     {GRAPHSAGE_OUT_DIM}")
print(f"  GraphSAGE num layers:     {GRAPHSAGE_NUM_LAYERS}")
print(f"  Dropout:                  {GRAPHSAGE_DROPOUT}")
print(f"  Learning rate:            {LEARNING_RATE}")
print(f"  Weight decay:             {WEIGHT_DECAY}")
print(f"  Num epochs:               {NUM_EPOCHS}")
print(f"  Batch size:               {BATCH_SIZE}")
print(f"  Positive class weight:    {POSITIVE_CLASS_WEIGHT}")
print(f"  Random seed:              {RANDOM_SEED}")
print(f"  Gradient clip max norm:   {GRAD_CLIP_MAX_NORM}")
print(f"  Risk threshold:           {RISK_THRESHOLD}")
print("=" * 50)
print(f"\nData sources (from Drive):")
print(f"  Graph folders: {GRAPHS_FOLDER}")
print(f"  Labels CSV:    {LABELS_CSV}")

## 5. Dataset Loading & Inspection

In [ ]:
# @title Load ClinicalGraphDataset from Drive-hosted graph files
# Discovers all *_graph.pt files across graph folders and attaches
# readmission labels from corpus_labels.csv.

dataset = ClinicalGraphDataset(
    graphs_folder=GRAPHS_FOLDER,
    labels_csv=LABELS_CSV,
)

print(f"Dataset size: {len(dataset)} graphs")

if len(dataset) == 0:
    print("\n*** WARNING: Dataset is empty! ***")
    print("Check that graph folders on Drive contain *_graph.pt files:")
    for folder in GRAPHS_FOLDER:
        p = Path(folder)
        exists = p.exists()
        n_pt   = len(list(p.glob("*_graph.pt"))) if exists else 0
        print(f"  {folder} — exists={exists}, *_graph.pt files={n_pt}")
    print(f"\nCheck labels CSV: {LABELS_CSV} — exists={Path(LABELS_CSV).exists()}")

In [ ]:
# @title Inspect a sample graph
if len(dataset) > 0:
    sample = dataset[0]
    print("Sample graph structure:")
    print(f"  Node types: {sample.node_types}")
    print(f"  Edge types: {sample.edge_types}")
    print()
    for nt in sample.node_types:
        n = sample[nt].num_nodes
        feat_shape = sample[nt].x.shape if hasattr(sample[nt], "x") else "N/A"
        print(f"  {nt}: {n} nodes, x shape = {feat_shape}")
    print()
    for et in sample.edge_types:
        n = sample[et].edge_index.size(1)
        print(f"  {et}: {n} edges")
    print(f"\n  y_readmission = {sample.y_readmission.item():.0f}")

In [ ]:
# @title Class balance analysis
if len(dataset) > 0:
    labels = []
    for i in range(len(dataset)):
        labels.append(int(dataset[i].y_readmission.item()))

    pos = sum(labels)
    neg = len(labels) - pos
    print(f"Total graphs:     {len(labels)}")
    print(f"Readmitted (1):   {pos} ({100*pos/len(labels):.1f}%)")
    print(f"Not readmitted:   {neg} ({100*neg/len(labels):.1f}%)")
    if pos > 0:
        print(f"Class imbalance:  1:{neg/pos:.1f}")
    else:
        print("  No positive labels!")

    # Bar plot
    fig, ax = plt.subplots()
    ax.bar(["No Readmission (0)", "Readmission (1)"], [neg, pos], color=["steelblue", "coral"])
    ax.set_ylabel("Number of Patients")
    ax.set_title("Readmission Label Distribution")
    for i, v in enumerate([neg, pos]):
        ax.text(i, v + max(neg, pos)*0.01, str(v), ha="center", fontweight="bold")
    plt.show()

## 6. Model Architecture

In [ ]:
# @title Build model & print parameter summary
# Set seed for reproducibility before model init
_set_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Build ClinicalGraphSAGE encoder
graph_encoder = ClinicalGraphSAGE(
    in_dim=GRAPHSAGE_IN_DIM,           # 768 — ClinicalBERT [CLS]
    hidden_dim=GRAPHSAGE_HIDDEN_DIM,   # 256
    out_dim=GRAPHSAGE_OUT_DIM,         # 256
    num_layers=GRAPHSAGE_NUM_LAYERS,   # 2
    dropout=GRAPHSAGE_DROPOUT,         # 0.3
)

# Build full model = encoder + readmission head
model = ReadmissionRiskModel(graph_encoder).to(device)

# Parameter breakdown
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel: ReadmissionRiskModel")
print(f"  ├── ClinicalGraphSAGE  (768→256→256)")
print(f"  │   Total params: {sum(p.numel() for p in graph_encoder.parameters()):,}")
print(f"  └── ReadmissionHead    (256→128→1)")
print(f"      Total params: {sum(p.numel() for p in model.readmission_head.parameters()):,}")
print(f"\n  Total parameters:    {total:,}")
print(f"  Trainable parameters: {trainable:,}")
print(f"  (~{total/1e6:.1f}M params — trained fully, no LoRA)")

In [ ]:
# @title Test forward pass with a single batch
if len(dataset) >= BATCH_SIZE:
    # Grab a small batch for shape verification
    small_set = Subset(dataset, list(range(min(BATCH_SIZE, len(dataset)))))
    loader = DataLoader(small_set, batch_size=min(BATCH_SIZE, len(dataset)), collate_fn=collate_fn)
    batch = next(iter(loader)).to(device)

    # BUG-FIX: PyG Batch._batch_dict is a property that calls collect('_batch'),
    # which raises KeyError when internal stores lack the '_batch' attribute.
    # Build batch_dict manually from per-node-type .batch vectors instead.
    batch_dict = {}
    for nt in batch.node_types:
        if hasattr(batch[nt], "batch"):
            batch_dict[nt] = batch[nt].batch
    if not batch_dict:
        batch_dict = None
    logits = model(batch.x_dict, batch.edge_index_dict, batch_dict)

    print(f"Batch size:          {min(BATCH_SIZE, len(dataset))}")
    print(f"Entity nodes:        {batch['entity'].num_nodes}")
    print(f"Logits shape:        {logits.shape}  (expected: [{min(BATCH_SIZE, len(dataset))}, 1])")
    print(f"Logits (raw):        {logits.squeeze().detach().cpu().tolist()}")
    print(f"After sigmoid:       {torch.sigmoid(logits).squeeze().detach().cpu().tolist()}")
    print("\n✓ Forward pass successful")
else:
    print(f"Not enough graphs for a batch of {BATCH_SIZE} — dataset has {len(dataset)} graphs")

## 7. Train/Val/Test Split

In [ ]:
# @title Split dataset (80/10/10) and create DataLoaders
# Guard: _split_dataset raises ValueError if dataset is empty
if len(dataset) == 0:
    raise SystemExit(
        "Cannot split — dataset is empty. "
        "Check that graph folders on Drive contain *_graph.pt files "
        "and that corpus_labels.csv has matching patient_id entries."
    )

_set_seed(RANDOM_SEED)

train_set, val_set, test_set = _split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train graphs: {len(train_set)}  →  {len(train_loader)} batches")
print(f"Val graphs:   {len(val_set)}    →  {len(val_loader)} batches")
print(f"Test graphs:  {len(test_set)}   →  {len(test_loader)} batches")

## 8. Training Loop

In [ ]:
# @title Setup optimizer, loss, scheduler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Re-init model for a clean training start
_set_seed(RANDOM_SEED)
graph_encoder = ClinicalGraphSAGE()
model = ReadmissionRiskModel(graph_encoder).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# BCEWithLogitsLoss applies sigmoid internally — no sigmoid in model forward
pos_weight = torch.tensor(POSITIVE_CLASS_WEIGHT, device=device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Optimizer:   AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Loss:        BCEWithLogitsLoss (pos_weight={POSITIVE_CLASS_WEIGHT})")
print(f"Scheduler:   CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"Device:      {device}")

In [ ]:
# @title Run training loop (checkpoint saved to Drive)
train_losses = []
val_aurocs   = []
best_val_auroc = 0.0

# Checkpoint directory on Drive so it persists across Colab sessions
checkpoint_dir = Path(CHECKPOINT_DIR)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    # ── Train one epoch ────────────────────────────────────────
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)

    # ── Validate ────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, device)
    val_auroc = val_metrics["readmission_auroc"]
    val_aurocs.append(val_auroc)

    # ── Step scheduler ──────────────────────────────────────────
    scheduler.step()

    # ── Logging ─────────────────────────────────────────────────
    lr_now = scheduler.get_last_lr()[0]
    marker = ""
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        marker = "  ★ BEST"

        # Save checkpoint to Drive
        ckpt_path = checkpoint_dir / "best_readmission_model.pt"
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_auroc": val_auroc,
            "train_loss": train_loss,
        }, str(ckpt_path))

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"train_loss: {train_loss:.4f} | "
          f"val_AUROC: {val_auroc:.4f} | "
          f"lr: {lr_now:.2e}{marker}")

print(f"\n✓ Training complete. Best val_AUROC: {best_val_auroc:.4f}")
print(f"Checkpoint saved to Drive: {checkpoint_dir / 'best_readmission_model.pt'}")

## 9. Training Curves

In [ ]:
# @title Plot loss and AUROC curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, len(train_losses)+1), train_losses, marker="o", color="steelblue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss (BCEWithLogitsLoss)")
ax1.grid(True, alpha=0.3)

# AUROC curve
ax2.plot(range(1, len(val_aurocs)+1), val_aurocs, marker="s", color="coral")
ax2.axhline(y=0.5, color="gray", linestyle="--", label="Random (0.5)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation AUROC")
ax2.set_title("Validation AUROC")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Final Test Evaluation

In [ ]:
# @title Evaluate best checkpoint on test set
# Reload best model from Drive checkpoint
best_ckpt_path = checkpoint_dir / "best_readmission_model.pt"

if best_ckpt_path.exists():
    graph_encoder = ClinicalGraphSAGE()
    model = ReadmissionRiskModel(graph_encoder).to(device)

    checkpoint = torch.load(str(best_ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"Saved val_AUROC: {checkpoint['val_auroc']:.4f}")

    test_metrics = evaluate(model, test_loader, device)
    print(f"\n*** Test AUROC: {test_metrics['readmission_auroc']:.4f} ***")
else:
    print(f"Checkpoint not found at {best_ckpt_path} — skipping test evaluation")

## 11. Inference on a Single Patient

Run the trained model on any `*_graph.pt` file to get a readmission risk prediction.

In [ ]:
# @title Single-patient inference
from datetime import datetime, timezone
from layer4.pipeline import _load_model
from shared.utils import save_json

# Select a graph file to predict on — change to any *_graph.pt path on Drive
GRAPH_FILE = f"{DRIVE_BASE}/data/mtsamples_graphs/mtsamples_0000__allergic_rhinitis__graph.pt"

graph_path = Path(GRAPH_FILE)

if not graph_path.exists():
    # Try to find any graph file on Drive
    candidates = []
    for folder in GRAPHS_FOLDER:
        candidates.extend(Path(folder).glob("*_graph.pt"))
    if candidates:
        graph_path = candidates[0]
        print(f"Using first available graph: {graph_path}")
    else:
        print("No graph files found — cannot run inference")
        graph_path = None

if graph_path is not None and best_ckpt_path.exists():
    # Load model from best checkpoint
    model = _load_model(str(best_ckpt_path), device)

    # Load graph
    from torch_geometric.data import HeteroData
    graph: HeteroData = torch.load(str(graph_path), weights_only=False)
    graph = graph.to(device)

    # Get patient_id
    try:
        patient_id = graph["patient"].patient_id
    except (AttributeError, KeyError):
        patient_id = graph_path.stem.replace("_graph", "")

    print(f"Patient ID: {patient_id}")
    print(f"Graph nodes: {sum(graph[nt].num_nodes for nt in graph.node_types)}")

    # Forward pass
    with torch.no_grad():
        logit = model(graph.x_dict, graph.edge_index_dict)  # [1, 1] raw logit
        prob = torch.sigmoid(logit)
        risk_score = float(prob.squeeze().cpu())

    risk_level = "HIGH" if risk_score >= RISK_THRESHOLD else "LOW"

    print(f"\nPrediction:")
    print(f"  Readmission risk: {risk_score:.4f}")
    print(f"  Risk level:       {risk_level}  (threshold={RISK_THRESHOLD})")

    # Save prediction JSON to Drive
    prediction = {
        "metadata": {
            "patient_id": patient_id,
            "layer": "layer4_readmission_risk_prediction",
            "predicted_at": datetime.now(timezone.utc).isoformat(),
        },
        "readmission_risk": round(risk_score, 4),
        "risk_level": risk_level,
    }

    output_dir = Path(PREDICTIONS_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{patient_id}_predictions.json"
    save_json(prediction, str(output_path))
    print(f"  Saved to Drive: {output_path}")
elif not best_ckpt_path.exists():
    print("No checkpoint — train the model first (Section 8).")

## 12. Batch Inference (All Graphs on Drive)

Runs the full [`run_pipeline()`](src/layer4/pipeline.py:94) on all discovered graphs.

In [ ]:
# @title Run batch inference over all graph folders on Drive
from layer4.pipeline import run_pipeline

if best_ckpt_path.exists():
    summary = run_pipeline(
        input_folder=GRAPHS_FOLDER,
        output_folder=PREDICTIONS_DIR,
        checkpoint_path=str(best_ckpt_path),
    )

    print(f"\nBatch inference complete:")
    print(f"  Processed: {summary['processed']}")
    print(f"  Failed:    {summary['failed']}")
    if summary["errors"]:
        print(f"\nErrors:")
        for err in summary["errors"]:
            print(f"  - {err}")
else:
    print("No checkpoint — train the model first (Section 8).")

## 13. Export Model for Deployment

Save a standalone model checkpoint (weights only) to Drive.

In [ ]:
# @title Export model weights only to Drive
EXPORT_PATH = f"{CHECKPOINT_DIR}/best_readmission_model.pt"

graph_encoder = ClinicalGraphSAGE()
model = ReadmissionRiskModel(graph_encoder)

if best_ckpt_path.exists():
    checkpoint = torch.load(str(best_ckpt_path), map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    torch.save({
        "model_state_dict": model.state_dict(),
        "epoch": checkpoint["epoch"],
        "val_auroc": checkpoint["val_auroc"],
        "config": {
            "in_dim": GRAPHSAGE_IN_DIM,
            "hidden_dim": GRAPHSAGE_HIDDEN_DIM,
            "out_dim": GRAPHSAGE_OUT_DIM,
            "num_layers": GRAPHSAGE_NUM_LAYERS,
            "dropout": GRAPHSAGE_DROPOUT,
            "risk_threshold": RISK_THRESHOLD,
        },
    }, EXPORT_PATH)
    print(f"Model exported to Drive: {EXPORT_PATH}")
    print(f"  epoch={checkpoint['epoch']}, val_AUROC={checkpoint['val_auroc']:.4f}")
else:
    print(f"No checkpoint to export at {best_ckpt_path}")

---

## Drive Folder Structure Reference

Before running, upload these to `MyDrive/CHARTA/`:

| Resource | Drive Path |
|---|---|
| Config (hyperparams) | `CHARTA/src/layer4/config.py` |
| GraphSAGE model | `CHARTA/src/layer4/graph_model.py` |
| Readmission head | `CHARTA/src/layer4/readmission_head.py` |
| Trainer (train/eval) | `CHARTA/src/layer4/trainer.py` |
| Dataset class | `CHARTA/src/layer4/clinical_dataset.py` |
| Inference pipeline | `CHARTA/src/layer4/pipeline.py` |
| Shared utils | `CHARTA/src/shared/utils.py` |
| Graph `.pt` files | `CHARTA/data/mtsamples_graphs/*_graph.pt` |
| Labels CSV | `CHARTA/data/corpus_labels.csv` |
| → Checkpoint saved to | `CHARTA/models/best_readmission_model.pt` |
| → Predictions saved to | `CHARTA/data/predictions/` |